# Tutorial 2 — FOL Formulas: Connectives & Quantifiers

The denotation of a formula built with connectives (¬, ∧, ∨, →, ↔) and quantifiers (∀, ∃) is computed by fuzzy operators applied to the truth degrees of its subformulas. These operators are stored in a `Logic` object, `interp.logic`, which is part of the interpretation. A formula is therefore parsed once against the signature, and the operators are chosen when its denotation is computed.

Each connective and quantifier can be written in several equivalent ways.

| Pattern | String syntax |
|---|---|
| Negation | `¬IsDog(x)`, `not IsDog(x)`, `~IsDog(x)` |
| Conjunction | `IsDog(x) ∧ IsCat(x)`, `&`, `and` |
| Disjunction | `IsDog(x) ∨ IsCat(x)`, `or` |
| Implication | `IsDog(x) → IsLarge(x)`, `->`, `=>` |
| Equivalence | `IsDog(x) ↔ ¬IsCat(x)`, `<->` |
| Universal | `∀ x: IsDog(x)`, `forall x: IsDog(x)` |
| Existential | `∃ x: IsDog(x)`, `exists x: IsDog(x)` |
| Diagonal (paired) | `∀ (x, y): Similar(x, y)` |
| Guarded | `∀ x \| IsLarge(x): IsDog(x)` |

The symbol `|` separates a guard from the body of a quantifier and does not denote disjunction. `IsDog(x) | IsCat(x)` is therefore a parse error, and disjunction is written `or` or `∨`.

In [ ]:
import torch
import sltn
from sltn import Signature, Interpretation, Logic

torch.manual_seed(0)
device = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
print("device:", device)

## Setup

An animal is a vector in ℝ² of two features, its **size** and its **furriness**, as in Tutorial 1. `IsDog` and `IsCat` have truth degree 1 at a dog prototype and a cat prototype, respectively, and decrease with the distance from it. `Similar` decreases with the distance between two animals.

In [ ]:
sig = Signature("animals")
sig.sort("Animal")
sig.variable("x", "Animal")
sig.variable("y", "Animal")
sig.predicate("IsDog", ["Animal"])
sig.predicate("IsCat", ["Animal"])
sig.predicate("Similar", ["Animal", "Animal"])
print(sig)

interp = Interpretation(sig).to(device)
interp["Animal"] = sltn.Type("R2", shape=(2,))   # (size, furriness)

# fixed predicates: truth degree 1 at the prototype, decreasing with the distance
dog = torch.tensor([0.0, 0.0], device=device)
cat = torch.tensor([1.0, 1.0], device=device)
interp["IsDog"]   = lambda a: torch.exp(-torch.norm(a - dog, dim=-1, keepdim=True))
interp["IsCat"]   = lambda a: torch.exp(-torch.norm(a - cat, dim=-1, keepdim=True))
interp["Similar"] = lambda a, b: torch.exp(-torch.norm(a - b, dim=-1, keepdim=True))

# 8 animals for x, 6 for y
interp["x"] = torch.randn(8, 2, device=device)
interp["y"] = torch.randn(6, 2, device=device)

print(interp)

## 1 — Connectives

Each connective and quantifier is computed by an operator of `interp.logic`. Printing it lists the operators in use.

In [ ]:
print(interp.logic)

These are the default operators of every interpretation, the **stable product configuration**.

- `∧` is the product `a·b`, `∨` is the probabilistic sum `a + b − a·b`, and `¬` is `1 − a`.
- `→` is the **Reichenbach** implication `1 − a + a·b`, as in LTN. It equals 1 only when `a = 0` or `b = 1`, so even `a → a` is below 1, e.g. 0.75 at `a = 0.5`.
- `↔` is `(a → b) ∧ (b → a)`, computed with the two operators above.
- `∀` and `∃` are p-means with p = 2 (section 2). `SatAgg` combines the clauses of a knowledge base.
- **`stable=True`** keeps the inputs at least `eps = 1e-6` away from 0 and 1, where some operators have an infinite or undefined gradient. The values change by at most `eps`. Every operator is stable by default, including operators constructed explicitly (section 6).

With `x` free, the denotation of each formula below contains **one truth degree per animal** and has shape `[8, 1]`. The table lists the first 4 animals. Every column is computed, animal by animal, from the columns `IsDog` and `IsCat`.

In [ ]:
formulas = {
    "IsDog":         "IsDog(x)",
    "IsCat":         "IsCat(x)",
    "¬IsDog":        "not IsDog(x)",
    "IsDog∧IsCat":   "IsDog(x) & IsCat(x)",       # a dog and a cat at once: low
    "IsDog∨IsCat":   "IsDog(x) ∨ IsCat(x)",
    "IsDog→¬IsCat":  "IsDog(x) → not IsCat(x)",   # a dog is not a cat
    "IsDog↔¬IsCat":  "IsDog(x) ↔ not IsCat(x)",
}
values = {label: interp(formula).value for label, formula in formulas.items()}
print("shape of each result:", list(values["IsDog"].shape))

print("\nanimal " + "".join(f"{label:>14s}" for label in formulas))
for i in range(4):
    print(f"{i:>6d} " + "".join(f"{v[i].item():>14.3f}" for v in values.values()))

## 2 — Quantifiers

`∀ x: φ(x)` aggregates the truth degrees of φ over the axis of `x` with **pMeanError**, p = 2, that is, `1 − (mean (1 − φ)²)^½`. The individuals with the lowest truth degrees have the largest effect, so a few counterexamples lower the result considerably. `∃ x: φ(x)` uses **pMean**, p = 2, that is, `(mean φ²)^½`, on which the individuals with the highest truth degrees have the largest effect. A larger p brings the two aggregators closer to min and max. The axis of `x` is removed, and the denotation is a single truth degree.

A quantifier removes only the axis of its own variable. The denotation of `Similar(x, y)` has one truth degree per pair, on the axes `x` (8 animals) and `y` (6 animals). Quantifying over one variable leaves the axis of the other, and quantifying over both gives a single truth degree.

In [ ]:
r_all = interp("forall x: IsDog(x)")    # every animal is a dog
r_ex  = interp("exists x: IsDog(x)")    # some animal is a dog
print("∀ x: IsDog(x) :", round(r_all.value.item(), 4), "  shape:", list(r_all.value.shape))
print("∃ x: IsDog(x) :", round(r_ex.value.item(),  4), "  shape:", list(r_ex.value.shape))

print()
for formula in [
    "Similar(x, y)",                      # one truth degree per pair (x, y)
    "forall x: Similar(x, y)",            # per y: is it similar to every x?
    "exists y: Similar(x, y)",            # per x: is it similar to some y?
    "forall x: exists y: Similar(x, y)",  # every x is similar to some y
]:
    r = interp(formula)
    print(f"{formula:<36s} axes {[a.name for a in r.axes]}   shape {list(r.value.shape)}")

A quantifier applies only to the smallest expression that follows it. `∀ x: IsDog(x) → ¬IsCat(x)` therefore means `(∀ x: IsDog(x)) → ¬IsCat(x)`, in which the second `x` is free, and not "no dog is a cat", which is written `∀ x: (IsDog(x) → ¬IsCat(x))`. sLTN emits a warning when a variable occurs both quantified and free, as in the first formula below.

In [ ]:
no_parens = interp("forall x: IsDog(x) -> not IsCat(x)")     # (forall x: IsDog(x)) -> not IsCat(x)
scoped    = interp("forall x: (IsDog(x) -> not IsCat(x))")   # no dog is a cat

print("forall x: IsDog(x) -> not IsCat(x)   -> shape:", list(no_parens.value.shape))   # x is still free
print("forall x: (IsDog(x) -> not IsCat(x)) -> shape:", list(scoped.value.shape))      # one truth degree

### Aliases

An **alias** of `x`, written `x1`, `x2` or `x'`, is a variable with the sort and the individuals of `x` but with an axis of its own. It is created on first use, so the individuals of `x` can be compared with each other without declaring and grounding a second variable.

The predicate `Bigger(a, b)`, "a is bigger than b", compares the sizes of two animals, i.e. their first features.

- The denotation of `Bigger(x, x1)` has axes `x` and `x1`, with 8 × 8 truth degrees, one for each pair of animals.
- The denotation of `Bigger(x, x)` has the single axis `x`, with 8 truth degrees, one for each animal compared with itself. The grounding is applied to 8 pairs instead of 64.

The second form expresses axioms on an individual and itself, such as irreflexivity, "no animal is bigger than itself", `∀ x: ¬Bigger(x, x)`. A grounding does not satisfy such an axiom automatically. The smooth comparison below gives 0.5 for two equal sizes, so the axiom has truth degree about 0.5. With a margin of 1, `Bigger(a, b)` is high only when `a` is larger than `b` by about 1, and the axiom holds. For a trainable `Bigger`, adding the axiom to the knowledge base is what enforces it during training.

In [ ]:
sig.predicate("Bigger", ["Animal", "Animal"])
interp["Bigger"] = lambda a, b: torch.sigmoid(5 * (a[:, :1] - b[:, :1]))   # smooth "size of a > size of b"

for formula in ["Bigger(x, x1)", "Bigger(x, x)"]:
    r = interp(formula)
    print(f"{formula:<14s} axes {[a.name for a in r.axes]}   shape {list(r.value.shape)}")

print("\nBigger(x, x):", interp("Bigger(x, x)").value[:, 0])
print(f"∀ x: ¬Bigger(x, x), smooth comparison : {interp('forall x: not Bigger(x, x)').value.item():.3f}")

# With a margin of 1: a must be bigger than b by about 1 or more
margin = interp.copy()
margin["Bigger"] = lambda a, b: torch.sigmoid(5 * (a[:, :1] - b[:, :1] - 1))
print(f"∀ x: ¬Bigger(x, x), with a margin     : {margin('forall x: not Bigger(x, x)').value.item():.3f}")

## 3 — Diagonal quantification

`∀ (x, y): φ(x, y)` pairs the variables positionally. The `i`-th individual of `x` is combined only with the `i`-th individual of `y`, instead of with every individual of `y`. Both variables must therefore have the same number of individuals.

In the following cell `y[i]` is a noisy copy of `x[i]`, for instance a second photo of the same animal. "Every animal is similar to its own photo" is the diagonal formula `∀ (x, y): Similar(x, y)`, and its truth degree is high. The Cartesian formula `∀ x: ∀ y: Similar(x, y)` compares every animal with every photo, and its truth degree is low.

In [ ]:
interp["x"] = torch.randn(8, 2, device=device)
interp["y"] = interp["x"] + 0.1 * torch.randn(8, 2, device=device)   # y[i]: a photo of x[i]

print("∀ (x, y): Similar(x, y)   [paired]   :", round(interp("∀ (x, y): Similar(x, y)").value.item(), 4))
print("∀ x: ∀ y: Similar(x, y)   [Cartesian]:", round(interp("∀ x: ∀ y: Similar(x, y)").value.item(), 4))

Pairing variables with different numbers of individuals is an error. In the following cell `y` is grounded with 6 animals in `uneven`, a copy of `interp`, so that `interp` keeps 8 animals for `y`. The same check applies to `∃ (x, y)`.

In [ ]:
uneven = interp.copy()
uneven["y"] = torch.randn(6, 2, device=device)
try:
    uneven("∀ (x, y): Similar(x, y)")
except ValueError as e:
    print("ValueError:", e)

## 4 — Guarded quantification

`∀ x | G(x): φ(x)` aggregates φ only over the individuals selected by the guard `G`. For instance, `∀ x | IsLarge(x): IsDog(x)` reads "every large animal is a dog". The truth degrees of the guard are **weights** in the aggregation.

- An individual with `G(x) = 0` is excluded, an individual with `G(x) = 1` counts fully, and an intermediate value counts partially.
- With a crisp guard, whose values are 0 or 1, the result equals the quantifier applied to the selected individuals only, as the following cell shows.
- Only the relative weights matter. A guard of 0.01 on every individual gives the same result as a guard of 1 on every individual.
- If the guard is 0 on every individual, `∀` gives 1 and `∃` gives 0, whatever the aggregator.

Guards are useful for **partially labelled** data, since unlabelled individuals can be excluded. A guard is grounded like any other predicate, as a function of its input. A tensor computed in advance from `interp["x"]` would no longer match once `x` is grounded with other data, and it would fail on `IsLarge(y)`, since `y` has a different number of individuals.

In [ ]:
sig.predicate("IsLarge", ["Animal"])
interp["IsLarge"] = lambda a: (a[:, :1] > 0).float()   # 1 if the size is positive, computed from the input

print("∀ x | IsLarge(x): IsDog(x)   :", round(interp("∀ x | IsLarge(x): IsDog(x)").value.item(), 4))

# ∀ applied to the large animals only: the same value
large = interp.copy()
large["x"] = interp["x"][interp["x"][:, 0] > 0]
print("∀ x: IsDog(x), large x only  :", round(large("∀ x: IsDog(x)").value.item(), 4),
      f"  ({len(large['x'])} of {len(interp['x'])} animals)")

## 5 — Definitions

A **definition**, declared with `sig.define(name, params, body)`, introduces a symbol whose meaning is given by a formula or a term written with the symbols of the signature. With a formula as body it defines a predicate, and with a term as body it defines a function. When `sig.parse` reads `name(t1, …, tn)`, it replaces it with `body`, in which each parameter `params[i]` is replaced by `ti`. A defined symbol is therefore expanded before any denotation is computed, and it requires **no grounding**.

The following cell defines two predicates. `DogNotCat(z)` stands for `IsDog(z) & not IsCat(z)`, and `SimilarToAll(z)` stands for `forall y: Similar(z, y)`, "z is similar to every animal of `y`". The printed formulas are the expansions produced by the parser.

In [ ]:
sig.define("DogNotCat", ["z"], "IsDog(z) & not IsCat(z)")
sig.define("SimilarToAll", ["z"], "forall y: Similar(z, y)")

for formula in ["DogNotCat(x)", "SimilarToAll(x)", "forall x: (DogNotCat(x) -> SimilarToAll(x))"]:
    print(f"{formula:<44s} parsed as  {sig.parse(formula)}")

print("\nDogNotCat(x), first 4 animals:", interp("DogNotCat(x)").value[:4, 0])

The variable `y` bound in the body of `SimilarToAll` is renamed when the argument is itself `y`. `SimilarToAll(y)` is therefore expanded to `∀ y': Similar(y, y')`, "the animal `y` is similar to every animal", and not to `∀ y: Similar(y, y)`.

A term as body defines a function. In the following cell the function `shrink` halves the size of an animal, and the defined function `quarter(z)` stands for `shrink(shrink(z))`. The output sort of `quarter` is the sort of its body, `Animal`, so `quarter(x)` can be the argument of `IsDog`.

In [ ]:
print(sig.parse("SimilarToAll(y)"))

sig.function("shrink", ["Animal"], "Animal")
interp["shrink"] = lambda a: a * torch.tensor([0.5, 1.0], device=device)   # halves the size, keeps the furriness

sig.define("quarter", ["z"], "shrink(shrink(z))")
print(sig.parse("IsDog(quarter(x))"))
print("IsDog(quarter(x)), first 4 animals:", interp("IsDog(quarter(x))").value[:4, 0])

## 6 — Changing the operators

`interp.logic` is a mutable `Logic`. Its operators are read and assigned like the entries of a dictionary, with the logical symbols as keys: `∧` / `&` / `and`, `∨` / `or`, `¬` / `not`, `→` / `->`, `↔` / `<->`, `∀` / `forall`, `∃` / `exists`, and `SatAgg`.

- An operator changed on a **copy** of the interpretation leaves the original unchanged.
- `interp.with_updates(logic=Logic(...))` returns a copy of the interpretation with another `Logic`, and the original keeps its operators. For instance, `interp.with_updates(logic=Logic.classical())` evaluates formulas with classical operators.
- An explicitly constructed operator is **stable by default**. `stable=False` gives the exact operator.
- `interp.logic = Logic()` restores every default operator.

In [ ]:
import sltn.interpretation.operators as ops

# A copy with a stricter ∀ (p = 4): the original is unchanged
strict = interp.copy()
strict.logic["∀"] = ops.AggregPMeanError(p=4)
print("∀ in interp :", interp.logic["∀"])
print("∀ in strict :", strict.logic["∀"])

phi = sig.parse("forall x: IsDog(x)")
print(f"forall x: IsDog(x)  ->  interp {interp(phi).value.item():.4f}   strict {strict(phi).value.item():.4f}")

# In place: Gödel conjunction and Goguen implication (the residuum of the product)
interp.logic["&"] = ops.AndMin()
interp.logic["->"] = ops.ImpliesGoguen()
print("\n∧ :", interp.logic["&"])
print("→ :", interp.logic["->"])

# Back to the defaults
interp.logic = Logic()
print("\n→ after the reset:", interp.logic["->"])

---

**Next:** [Tutorial 3 — Fuzzy Operators & Gradients](3-FuzzyOperators&Gradients.ipynb) looks at *why* the choice of connective and aggregation operator matters for learning.